In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import FunctionTransformer
from bs4 import BeautifulSoup

In [3]:
from pathlib import Path

TRAIN_PATH = Path("../data/processed/train.csv")

train_df = pd.read_csv(TRAIN_PATH)

X_train = train_df["review"]
y_train = train_df["sentiment"]

print("Training reviews:", len(X_train))
print("Training labels:", y_train.value_counts())

Training reviews: 40000
Training labels: sentiment
positive    20000
negative    20000
Name: count, dtype: int64


In [4]:
def strip_html_batch(texts):
    return [
        BeautifulSoup(text, "html.parser").get_text(separator=" ")
        for text in texts
    ]


baseline = Pipeline([
    (
        "html",
        FunctionTransformer(
            strip_html_batch,
            validate=False
        )
    ),
    (
        "vectorizer",
        CountVectorizer()
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

print("Baseline pipeline created successfully!")

Baseline pipeline created successfully!


In [5]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    baseline,
    X_train,
    y_train,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

print("Fold accuracies:", np.round(scores, 4))
print(f"Mean CV accuracy: {scores.mean():.4f}")
print(f"Standard deviation: {scores.std():.4f}")

Fold accuracies: [0.883  0.8838 0.8886 0.8914 0.88  ]
Mean CV accuracy: 0.8854
Standard deviation: 0.0041


In [6]:
import re
import string

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import FunctionTransformer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from bs4 import BeautifulSoup


def strip_html_batch(texts):
    return [
        BeautifulSoup(text, "html.parser").get_text(separator=" ")
        for text in texts
    ]


def lowercase_batch(texts):
    return [text.lower() for text in texts]


def remove_punctuation_batch(texts):
    return [
        re.sub(r"[^\w\s]", " ", text)
        for text in texts
    ]


# Keep negation words to preserve sentiment
custom_stop_words = list(
    ENGLISH_STOP_WORDS - {"not", "no", "never"}
)


def remove_stopwords_batch(texts):
    cleaned = []

    for text in texts:
        words = text.split()
        words = [
            word for word in words
            if word.lower() not in custom_stop_words
        ]
        cleaned.append(" ".join(words))

    return cleaned


print("Preprocessing functions ready!")

Preprocessing functions ready!


In [7]:
from snowballstemmer import stemmer

english_stemmer = stemmer("english")

def stem_text_batch(texts):
    return [
        " ".join(english_stemmer.stemWords(str(text).lower().split()))
        for text in texts
    ]

print(stem_text_batch([
    "The movies were amazing and entertaining",
    "The performances were disappointing"
]))

['the movi were amaz and entertain', 'the perform were disappoint']


Raw text: 0.8869 ± 0.0013
HTML stripping: 0.8873 ± 0.0006
Lowercasing: 0.8873 ± 0.0006
Punctuation removal: 0.8873 ± 0.0006
Stop-word removal: 0.8800 ± 0.0027


,Variant,Mean CV Accuracy,Std Dev
0,Raw text,0.88685,0.001263
1,HTML stripping,0.88730,0.000640
2,Lowercasing,0.88730,0.000640
3,Punctuation removal,0.88730,0.000640
4,Stop-word removal,0.87995,0.002745


In [1]:
%pip install snowballstemmer

Note: you may need to restart the kernel to use updated packages.


Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "e:\LAB ASSIGNMENTS\.venv\Lib\site-packages\pip\__main__.py", line 22, in <module>
    from pip._internal.cli.main import main as _main
  File "e:\LAB ASSIGNMENTS\.venv\Lib\site-packages\pip\_internal\cli\main.py", line 11, in <module>
    from pip._internal.cli.autocompletion import autocomplete
  File "e:\LAB ASSIGNMENTS\.venv\Lib\site-packages\pip\_internal\cli\autocompletion.py", line 12, in <module>
    from pip._internal.cli.main_parser import create_main_parser
  File "e:\LAB ASSIGNMENTS\.venv\Lib\site-packages\pip\_internal\cli\main_parser.py", line 9, in <module>
    from pip._internal.build_env import get_runnable_pip
  File "e:\LAB ASSIGNMENTS\.venv\Lib\site-packages\pip\_internal\build_env\__init__.py", line 8, in <module>
    from pip._internal.build_env.installer import (
    ...<2 lines>...
    )
  File "e:\LAB ASSIGNMENTS\.ve

In [1]:
from snowballstemmer import stemmer

english_stemmer = stemmer("english")
print(english_stemmer.stemWords(["movies", "amazing", "entertaining"]))

['movi', 'amaz', 'entertain']


In [8]:

variants = {
    "Raw text": [],

    "HTML stripping": [
        ("html", FunctionTransformer(strip_html_batch, validate=False))
    ],

    "Lowercasing": [
        ("html", FunctionTransformer(strip_html_batch, validate=False)),
        ("lowercase", FunctionTransformer(lowercase_batch, validate=False))
    ],

    "Punctuation removal": [
        ("html", FunctionTransformer(strip_html_batch, validate=False)),
        ("lowercase", FunctionTransformer(lowercase_batch, validate=False)),
        ("punctuation", FunctionTransformer(remove_punctuation_batch, validate=False))
    ],

    "Stop-word removal": [
        ("html", FunctionTransformer(strip_html_batch, validate=False)),
        ("lowercase", FunctionTransformer(lowercase_batch, validate=False)),
        ("punctuation", FunctionTransformer(remove_punctuation_batch, validate=False)),
        ("stopwords", FunctionTransformer(remove_stopwords_batch, validate=False))
    ],

    "Stemming": [
        ("html", FunctionTransformer(strip_html_batch, validate=False)),
        ("lowercase", FunctionTransformer(lowercase_batch, validate=False)),
        ("punctuation", FunctionTransformer(remove_punctuation_batch, validate=False)),
        ("stemming", FunctionTransformer(stem_text_batch, validate=False))
    ]
}

In [9]:
# Compare raw counts against TF-IDF

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
import numpy as np

vectorizers = {
    "CountVectorizer": CountVectorizer(),
    "TF-IDF": TfidfVectorizer()
}

tfidf_results = {}

for name, vectorizer in vectorizers.items():

    model = Pipeline([
        (
            "html",
            FunctionTransformer(
                strip_html_batch,
                validate=False
            )
        ),
        ("vectorizer", vectorizer),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ])

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy",
        n_jobs=-1
    )

    tfidf_results[name] = scores

    print(f"\n{name}")
    print("Fold accuracies:", np.round(scores, 4))
    print(f"Mean accuracy: {scores.mean():.4f}")
    print(f"Standard deviation: {scores.std():.4f}")


CountVectorizer
Fold accuracies: [0.883  0.8838 0.8886 0.8914 0.88  ]
Mean accuracy: 0.8854
Standard deviation: 0.0041

TF-IDF
Fold accuracies: [0.8879 0.8921 0.8974 0.8988 0.8891]
Mean accuracy: 0.8931
Standard deviation: 0.0043


In [10]:
from snowballstemmer import stemmer

english_stemmer = stemmer("english")

def stem_text_batch(texts):
    return [
        " ".join(english_stemmer.stemWords(str(text).lower().split()))
        for text in texts
    ]

print(stem_text_batch([
    "The movies were amazing and entertaining",
    "The performances were disappointing"
]))

['the movi were amaz and entertain', 'the perform were disappoint']


In [11]:
# P05: TF-IDF vs. Bag-of-Words Counts

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
import numpy as np
import pandas as pd

vectorizers = {
    "Raw Counts": CountVectorizer(),
    "TF-IDF": TfidfVectorizer()
}

comparison_results = []

for name, vectorizer in vectorizers.items():

    model = Pipeline([
        (
            "html",
            FunctionTransformer(
                strip_html_batch,
                validate=False
            )
        ),
        ("vectorizer", vectorizer),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ])

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy",
        n_jobs=-1
    )

    comparison_results.append({
        "Representation": name,
        "Mean CV Accuracy": scores.mean(),
        "Standard Deviation": scores.std()
    })

    print(
        f"{name}: {scores.mean():.4f} "
        f"+/- {scores.std():.4f}"
    )

comparison_df = pd.DataFrame(comparison_results)
comparison_df

Raw Counts: 0.8854 +/- 0.0041
TF-IDF: 0.8931 +/- 0.0043


,Representation,Mean CV Accuracy,Standard Deviation
0,Raw Counts,0.88535,0.004092
1,TF-IDF,0.89305,0.004341


In [12]:
# P05: TF-IDF Unigrams vs Unigrams + Bigrams

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
import numpy as np
import pandas as pd

ngram_results = []

for name, ngram_range in [
    ("Unigrams", (1, 1)),
    ("Unigrams + Bigrams", (1, 2))
]:

    model = Pipeline([
        (
            "html",
            FunctionTransformer(
                strip_html_batch,
                validate=False
            )
        ),
        (
            "vectorizer",
            TfidfVectorizer(
                ngram_range=ngram_range,
                min_df=2,
                max_features=100000
            )
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ])

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy",
        n_jobs=-1
    )

    ngram_results.append({
        "Representation": name,
        "Mean CV Accuracy": scores.mean(),
        "Standard Deviation": scores.std()
    })

    print(
        f"{name}: {scores.mean():.4f} "
        f"+/- {scores.std():.4f}"
    )

ngram_df = pd.DataFrame(ngram_results)
ngram_df

Unigrams: 0.8930 +/- 0.0043
Unigrams + Bigrams: 0.9017 +/- 0.0032


,Representation,Mean CV Accuracy,Standard Deviation
0,Unigrams,0.893000,0.004321
1,Unigrams + Bigrams,0.901675,0.003222
